In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# convert mnist image files into tensors of 4 dimensions (no of images, height, width, color channel)

transforms = transforms.ToTensor()

## load data 

In [4]:
# traininig data

train_data = datasets.MNIST(root='cnn_data',train= True, download=True, transform=transforms)


100.0%
100.0%
100.0%
100.0%


In [6]:
test_data = datasets.MNIST(root='cnn_data',train= False, download=True, transform=transforms)

In [7]:
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: cnn_data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [8]:
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: cnn_data
    Split: Test
    StandardTransform
Transform: ToTensor()

## convoultion and pooling thorugh one image

In [10]:
# create a small batch size for images -> 10

train_loader = DataLoader(train_data, batch_size=10, shuffle=True)
test_loader = DataLoader(test_data, batch_size=10, shuffle=False)

In [11]:
#define convolutional model
#describe convolutional layer and what its doing (2 layers)
#example

conv1 = nn.Conv2d(1, 6, 3, 1)  #input, output, kernal, stride
conv2 = nn.Conv2d(6, 16, 3, 1)

In [16]:
#grab 1 mnist image
for i, (X_Train, y_train) in enumerate(train_data):
    break

In [17]:
X_Train.shape

torch.Size([1, 28, 28])

In [18]:
x = X_Train.view(1,1,28,28)

In [19]:
# perdom our firt convolution

x = F.relu(conv1(x)) # releu -> rectified linear unit (activation function) 


In [ ]:
# 1 is the image, 6 is the filters, 26 is the image size which reduced from 28x28 because of stide

x.shape

torch.Size([1, 6, 26, 26])

In [22]:
# pass through the pooling layer

x = F.max_pool2d(x, 2, 2) # kernel of 2 and stride of 2

In [ ]:
# 26/2 = 13     

x.shape 

torch.Size([1, 6, 13, 13])

In [24]:
# do our second convolutional layer

x = F.relu(conv2(x))

In [ ]:
x.shape 

torch.Size([1, 16, 11, 11])

In [26]:
# pooling layer 2

x = F.max_pool2d(x, 2, 2)

In [ ]:
# 11/5 -> 5.5 but we have to round down, becase we cant can round up to invent new data

x.shape

torch.Size([1, 16, 5, 5])

In [30]:
((28-2) / 2 - 2) / 2

5.5

## Model Class

In [31]:
# model class

class ConvolutionalNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 3, 1)
        self.conv2 = nn.Conv2d(6, 16, 3, 1)
        # fully connected layer
        self.fc1 = nn.Linear(5*5*16, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    #forward function
    def forward(self, x):
        #first pass
        X = F.relu(conv1(X))
        X = F.max_pool2d(X, 2, 2) # 2x2 kernel and stride 2
        #second pass
        X = F.relu(conv2(X))
        X = F.max_pool2d(X, 2, 2)

        # re view to flatten it out
        X = X.view(-1, 16*5*5) # negative so we can vary batch size

        # fully connected layers
        X = F.relu(self.fc1(x))
        X = F.relu(self. fc2(x))
        X = self.fc3(X)
        return F.log_softmax(X, dim=1)


In [32]:
# create instance of out model
torch.manual_seed(41)

In [33]:
model = ConvolutionalNetwork()
model

ConvolutionalNetwork(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [34]:
# loss function optmizer

criterion = nn.CrossEntropyLoss()
optmizer = torch.optim.Adam(model.parameters(), lr = 0.001)